Установка и импорт нужных библиотек

In [ ]:
!pip install bertopic gensim

In [ ]:
import re
import nltk
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from bertopic import BERTopic
from hdbscan import HDBSCAN

nltk.download("stopwords")
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

import spacy
nlp = spacy.load("en_core_web_sm", disable=["pos", "parser", "ner"])

import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
RANDOM_STATE = 42

# Тематическое моделирование: LDA, NMF и BERTopic

В этой практике мы рассмотрим и сравним друг с другом три метода тематического моделирования:
- **LDA (Latent Dirichlet Allocation)** — классический вероятностный метод.
- **NMF (Non-Negative Matrix Factorization)** — метод факторизации матриц.
- **BERTopic** — современный подход на основе эмбеддингов и кластеризации.

## Загрузка данных

Возьмём корпус **20 Newsgroups** — популярный датасет новостных постов, часто используемый при изучении классификации и кластеризации текстов.


In [ ]:
newsgroups = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
texts = newsgroups.data[:5000]  # для быстрой работы ограничимся 5000 документами

Взглянем на некоторые документы.

In [ ]:
for i, doc in enumerate(texts[:3], 1):
    print(f"Документ {i}:")
    print(doc[:500], "...")  # ограничим вывод 500 символами
    print()

На начальном этапе необходимо предобработать тексты:
- привести к нижнему регистру;
- убрать цифры и не буквенные символы;
- токенизировать и лемматизировать;
- отфильтровать стоп-слова, убрать слишком короткие токены.

Это минимальный набор действий, который необходимо реализовать в функции ниже, но вы можете добавить что-нибудь ещё.

In [ ]:
def preprocess(text):
    text = text.lower()
    text = ...  # регулярное выражение для удаления всех не-символов
    text = ...  # регулярное выражение для удаления всех цифр

    # можно воспользоваться моделью spacy для токенизации или лемматизации,
    # либо использовать любой другой удобный вам инструмент
    doc = ...
    # смотрим, чтобы леммы не были в списке стоп-слов И
    # длина токена была строго больше 2
    tokens = [
        token.lemma_
        for token in doc
        if (...)
    ]
    return " ".join(tokens)

Создадим список предобработанных текстов.

In [ ]:
texts_clean = [preprocess(t) for t in texts]

Посмотрим на некоторые из них.

In [ ]:
for i, doc in enumerate(texts_clean[:3], 1):
    print(f"Документ {i}:")
    print(doc[:500], "...")  # ограничим вывод 500 символами
    print()

## LDA и NMF

### Векторизация

Для моделей LDA и NMF мы используем разные способы преобразования текста в числовую матрицу.

- **Для LDA:** используем `CountVectorizer` (частоты слов).  
  LDA — вероятностная модель, которая работает с дискретными событиями. Слово считается как "событие", которое встречается определённое число раз в документе. Следовательно, для LDA лучше подавать сырые счётчики слов, а не нормализованные значения.

- **Для NMF:** используем `TfidfVectorizer` (взвешенные частоты).  
  NMF — метод матричной факторизации. TF-IDF помогает снизить влияние слишком частых слов (например, _the_, _and_) и выделить более информативные слова. Это делает темы более интерпретируемыми.

In [ ]:
# CountVectorizer для LDA
vectorizer_count = CountVectorizer(max_df=0.8, min_df=2)
X_count = vectorizer_count.fit_transform(texts_clean)

# TfidfVectorizer для NMF
vectorizer_tfidf = TfidfVectorizer(max_df=0.8, min_df=2)
X_tfidf = vectorizer_tfidf.fit_transform(texts_clean)

С помощью аргументов **max_df** и **min_df** можно фильтровать слова по частотности (подробнее — [здесь](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) и [здесь](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)). Попробуйте в дальнейшем поменять значения этих параметров и посмотрите, как это влияет на качество кластеризации.

### Обучаем модели

Важным шагом в тематическом моделировании является выбор количества тем. Слишком малое количество тем может привести к слишком общим и смешанным темам, а слишком большое — к избыточной фрагментации.

Для начала зададим небольшое значение.

In [ ]:
n_topics = 5

Теперь обучим модели на матрицах, полученных на этапе векторизации.

In [ ]:
# LDA
lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_STATE)
lda.fit(X_count)

# NMF
nmf = NMF(n_components=n_topics, random_state=RANDOM_STATE)
nmf.fit(X_tfidf)

Выведем результаты обучения.

In [ ]:
def print_top_words(model, feature_names, n_top_words=10):
  """Считает и выводит топ-слова каждой темы"""
  for topic_idx, topic in enumerate(model.components_, 1):
      message = f"Topic #{topic_idx}: "
      message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
      print(message)

In [ ]:
print("LDA topics")
print_top_words(lda, vectorizer_count.get_feature_names_out())

print("\nNMF topics")
print_top_words(nmf, vectorizer_tfidf.get_feature_names_out())

Посмотрите на топ-слова в каждой теме, оцените, насколько хорошо интерпретируются выделенные темы. Скорее всего с небольшим значением параметра **n_topics** темы плохо объясняются, поэтому поэкспериментируйте: задайте разное количество тем, оцените вручную, становится ли смысл тем более очевидным.

### Метрики оценивания

Качество кластеризации можно оценивать не только вручную, но и с помощью специальных метрик. Рассмотрим три такие метрики.

**1. Когерентность**

Когерентность оценивает семантическую связность слов внутри тем. Чем выше значение когерентности (ближе к 1), тем более осмысленными и интерпретируемыми являются темы.

Мы будем использовать готовую модель из библиотеки **gensim**. Перед этим необходимо веса из моделей **sklearn**, с которыми мы работаем.

In [ ]:
def calculate_coherence(model, vectorizer, texts_tokenized, n_top_words=10):
    """Вычисляет когерентность"""

    feature_names = vectorizer.get_feature_names_out()

    topics = []
    for topic_weights in model.components_:
        top_words = [feature_names[i] for i in topic_weights.argsort()[:-n_top_words-1:-1]]
        topics.append(top_words)

    dictionary = Dictionary(texts_tokenized)

    coherence_model = CoherenceModel(
        topics=topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v'
    )

    return coherence_model.get_coherence(), topics

In [ ]:
coherence_lda, lda_topics = calculate_coherence(
    lda, vectorizer_count, texts_tokenized
)
coherence_nmf, nmf_topics = calculate_coherence(
    nmf, vectorizer_tfidf, texts_tokenized
)

print(f"LDA когерентность: {coherence_lda:.4f}")
print(f"NMF когерентность: {coherence_nmf:.4f}")

**2. Перплексия**

Перплексия насколько модель “удивлена” данным, которые она не видела во время обучения. Следовательно, чем точнее модель может предсказывать вероятность слов в документе, тем ниже будет значение перплексии для неё.

Важно отметить, что перплексия может применяться только к вероятностным моделям, поэтому с ней не получится оценить алгоритм NMF.

Здесь удобно использовать встроенный метод для модели LDA из **sklearn**.

In [ ]:
lda_perplexity = lda.perplexity()

In [ ]:
print(lda_perplexity)

**3. Разнообразие тем**

Разнообразие измеряет, насколько уникальны слова в разных темах. Если все темы содержат одни и те же слова, разнообразие будет низким. Нам важно, чтобы модель кластеризации не создавала "клонов" тем.

Разнообразие считается как процент уникальных слов среди топ-N слов всех тем. Ниже допишите функцию для подсчёта этой метрики.

In [ ]:
def calculate_topic_diversity(model, feature_names, n_top_words=10):
  """Вычисляет разнообразие тем"""

  all_top_words = []

  for topic_idx, topic in enumerate(model.components_, 1):
      top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
      all_top_words.extend(top_words)

  unique_words = ... # нужно отобрать уникальные слова из списка топ-слов
  num_words = ... # затем их количество
  total_words = ... # теперь количество всех топ-слов

  diversity_score = ... # формула вычисления разнообразия

  return diversity_score

In [ ]:
calculate_topic_diversity(lda, vectorizer_count.get_feature_names_out())

In [ ]:
calculate_topic_diversity(nmf, vectorizer_tfidf.get_feature_names_out())

**Интепретация метрик**: к каким значениям мы хотим стремиться?

- Когерентность: значения выше 0.5 обычно считаются хорошими, выше 0.7 — отличными.
- Перплексия (Perplexity): чем ниже, тем лучше. Сравнивайте значения для разных моделей или разных гиперпараметров.
- Разнообразие (Diversity): чем ближе к 1, тем менее перекрываются слова между темами.

Метрики — удобный инструмент, *но не стоит полагаться исключительно на них*. Обязательно проводите ручную проверку тем:
- Просматривайте топ-слова каждой темы: они должны образовывать осмысленную группу.
- Ищите пересекающиеся темы: иногда метрики хорошие, но темы дублируют друг друга.
- Оценивайте практическую полезность: темы должны быть релевантны вашей задаче.

Почему это важно:
- Метрики могут быть обманчивы: например, высокая когерентность не всегда означает полезные темы.
- Только человек может оценить семантическую осмысленность и практическую ценность тем.
- Ручная проверка помогает выявить проблемы, которые не отражаются в числовых метриках.



Далее вы можете экспериментировать с различными параметрами, которые обсуждались ранее, чтобы получить наиболее оптимальные значения метрик.

## BERTopic

В этом разделе мы рассмотрим более современный алгоритм кластеризации на основе эмбеддингов. В этом задании мы воспользуемся для трансформером BERT для создания эмбеддингов и алгоритмом HDBSCAN для последующей группировки эмбеддингов. Для реализации этого пайплайна нам поможет библиотека **bertopic**.

### Алгоритм кластеризации

Мы будем использовать алгоритм HDBSCAN для кластеризации эмбеддингов документов.

Особенности работы HDBSCAN:
- Иерархическая кластеризация: Сначала строится иерархия возможных кластеров на основе взаимной достижимости точек.
- Основан на плотности: В отличие от некоторых других алгоритмов, как, например, k-means, HDBSCAN не требует задания числа кластеров заранее — он находит области с высокой плотностью точек.
- Устойчивость к шуму: Точки в разреженных областях помечаются как шум, что полезно для текстовых данных, где некоторые документы могут не принадлежать ни к одной теме.
- Разные размеры кластеров: Может находить кластеры разной формы и размера.

Таким образом, HDBSCAN оказывается полезным для задачи тематического моделирования благодаря тому, что нам не нужно самостоятельно задавать количество тем, а также у нас появляется возможность автоматически исключить шум в данных.

Воспользуемся моделью из библиотеки **hdbscan**.

In [ ]:
hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=15)

Немного о параметрах:
- `min_cluster_size` определяет минимальный размер кластера;
- `min_samples` влияет на плотность кластеров. Чем выше значение этого параметра, тем больше данных помечаются как шумные и убираются из кластера.

### Обучение модели BERTopic


BERTopic объединяет несколько этапов:

1. Создание эмбеддингов.
2. Их кластеризация.
3. Извлечение тем — определение ключевых слов для каждого кластера.
3. Постобработка — объединение похожих тем и фильтрация.

Обучение модели может занять некоторое время, это нормально :)

In [ ]:
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", hdbscan_model=hdbscan_model)
topics, probabilities = topic_model.fit_transform(texts_clean)

Примечание: `all-MiniLM-L6-v2` — эмбеддинговая модель по умолчанию в BERTopic. Вы можете подобрать любую другую модель, например, из библиотеки [sentence_transformers](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html) или поискать на Hugging Face.

Теперь мы можем вывести информацию о полученных темах. При запуске ячейки кода выведется таблица, в которой будет указано:
- `Topic`: id темы (-1 обычно означает шум)
- `Count`: количество документов в теме
- `Name`: название темы (на основе топ-слов)
- `Representation`: ключевые слова темы
- `Representative_Docs`: ключевые документы темы

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info)

Полученные результаты можно визуализировать. Вы увидите распределение тем в двумерном пространстве, что помогает увидеть близость/пересечение тем и обнаружить выбросы.



In [ ]:
fig = topic_model.visualize_topics()
fig.show()

Попробуйте обучить модель с другими параметрами кластеризации, сравните результаты.